In [6]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from autotsc.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV8Base
from autotsc.old_models import StackerV4, LokyStackerV5, LokyStackerV5SoftRF, LokyStackerV6
from autotsc import utils, data_loader
import polars as pl

In [ ]:
dataset = "Worms"
dataset = 'Car'
dataset = 'HandOutlines'
# dataset = 'UWaveGestureLibraryY'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
# print(X_train.dtype)
# print(y_train.dtype)

(60, 1, 577) (60,) (60, 1, 577) (60,)


In [3]:
#m = TSCGlue(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
#m.fit(X_train, y_train)
#y_pred = m.predict(X_test)

In [4]:
m = LokyStackerV8Base(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
m.fit(X_train, y_train)

[0.00s] Starting executor with 16 workers, run_dir=./tscglue/95d19e304e024456
[0.00s] Saved X and y to disk in 0.00s
[0.98s] Computed QUANT features (60, 5222) (2.39 MB) in 0.9758s
[1.10s] Starting repetition 0
[3.90s] Computed MultiRocket features (60, 49728) (22.76 MB) in 2.7997s
[6.75s] Computed Hydra features (60, 7168) (3.28 MB) in 2.8476s
[11.70s] Computed RDST features (60, 30000) (13.73 MB) in 4.9480s
[11.71s] Starting training with 16 workers for 40 models
[21.71s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.6605s for f-6/r-0 (2.85 MB)
[21.75s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.6896s for f-9/r-0 (2.85 MB)
[21.83s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.7821s for f-5/r-0 (2.85 MB)
[21.90s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8511s for f-8/r-0 (2.85 MB)
[21.92s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8804s for f-1/r-0 (2.85 MB)
[21.93s] Trained multirockethydra-bestk-p-ridgecv_r0 in 0.8910s for f-7/r-0 (2.85 MB)
[21.95s] Train

,random_state,492
,n_repetitions,1
,k_folds,10
,n_jobs,16
,keep_features,True
,verbose,2


In [7]:
preds = m.predict_per_model(X_test)
tests_accs = []
for model_name, model_preds in preds.items():
    acc = accuracy_score(y_test, model_preds)
    tests_accs.append({"model": model_name, "test_accuracy": acc})
test_df = pl.DataFrame(tests_accs)

[0.0000s] Starting prediction
Starting executor with 16 workers, run_dir=./tscglue/95d19e304e024456
[4.7060s] Computed features for prediction
[4.7490s] Feature arrays saved to mmap files
[4.7492s] Starting prediction with 16 workers for 40 first-level models
[20.9808s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.1959s
[21.0210s] Predicted quant-etc_r0 in 0.0255s
[21.0684s] Predicted quant-etc_r0 in 0.0250s
[21.1081s] Predicted quant-etc_r0 in 0.0254s
[21.1574s] Predicted quant-etc_r0 in 0.0256s
[21.2097s] Predicted rdst-p-ridgecv_r0 in 0.0475s
[21.2984s] Predicted rdst-p-ridgecv_r0 in 0.0659s
[21.3092s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.1165s
[21.3287s] Predicted rdst-p-ridgecv_r0 in 0.0345s
[21.3583s] Predicted rdst-p-ridgecv_r0 in 0.0240s
[21.3843s] Predicted rdst-p-ridgecv_r0 in 0.0229s
[21.4323s] Predicted rdst-p-ridgecv_r0 in 0.1033s
[21.4373s] Predicted rdst-p-ridgecv_r0 in 0.0389s
[21.4866s] Predicted rdst-p-ridgecv_r0 in 0.0508s
[21.5417s] Predicted rds

In [8]:
pl.DataFrame(m.summary()).join(test_df, on="model").sort("test_accuracy", descending=True)

model,level,oof_accuracy,test_accuracy
str,i64,f64,f64
"""probability-et""",1,0.916667,0.933333
"""probability-rf""",1,0.933333,0.933333
"""quant-etc_r0""",0,0.916667,0.916667
"""rdst-p-ridgecv_r0""",0,0.933333,0.916667
"""probability-ridgecv""",1,0.916667,0.916667
"""multirockethydra-bestk-p-ridge…",0,0.95,0.9
"""rstsf_r0""",0,0.866667,0.816667


In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
F = m.get_features()
F.estimated_size('gb')

In [ ]:
P = m.get_oof_predictions()
P.estimated_size('gb')

In [ ]:
m = LokyStackerV6(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
m = LokyStackerV5(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")